# Structured output
Structured output allows agents to return data in a specific, predictable format. i.e. 
- JSON Object
- Pydantic model
- Dataclass
LangChain’s create_agent handles structured output automatically.


## Response format
- Use response_format to control how the agent returns structured data:
    - ToolStrategy[StructuredResponseT]: Uses tool calling for structured output
    - ProviderStrategy[StructuredResponseT]: Uses provider-native structured output
    - type[StructuredResponseT]: Schema type - automatically selects best strategy based on model capabilities
    - None: Structured output not explicitly requested
- When a schema type is provided directly, LangChain automatically chooses:
    - ProviderStrategy if the model and provider chosen supports native structured output (e.g. OpenAI, Anthropic (Claude), or xAI (Grok)).
    - ToolStrategy for all other models.



### Provider strategy
Some model providers support structured output natively through their APIs
```
class ProviderStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    strict: bool | None = None
```
schema `required`
- The schema defining the structured output format. Supports:
    - Pydantic models: BaseModel subclasses with field validation. Returns validated Pydantic instance.
    - Dataclasses: Python dataclasses with type annotations. Returns dict.
    - TypedDict: Typed dictionary classes. Returns dict.
    - JSON Schema: Dictionary with JSON schema specification. Returns dict.

​
strict `Optional` 
- boolean parameter to enable strict schema adherence. Supported by some providers (e.g., OpenAI and xAI). Defaults to None (disabled).

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5.5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

### Tool Calling Strategy
- For models that don’t support native structured output, LangChain uses tool calling to achieve the same result. 
- This works with all models that support tool calling (most modern models).
```
class ToolStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    tool_message_content: str | None
    handle_errors: Union[
        bool,
        str,
        type[Exception],
        tuple[type[Exception], ...],
        Callable[[Exception], str],
    ]
```
schema `required`
- The schema defining the structured output format. Supports:
    - Pydantic models: BaseModel subclasses with field validation. Returns validated Pydantic instance.
    - Dataclasses: Python dataclasses with type annotations. Returns dict.
    - TypedDict: Typed dictionary classes. Returns dict.
    - JSON Schema: Dictionary with JSON schema specification. Returns dict.
    - Union types: Multiple schema options. The model will choose the most appropriate schema based on the context.

tool_message_content
- Custom content for the tool message returned when structured output is generated. If not provided, defaults to a message showing the structured response data.
​
handle_errors
- Error handling strategy for structured output validation failures. Defaults to True.
    - True: Catch all errors with default error template
    - str: Catch all errors with this custom message
    - type[Exception]: Only catch this exception type with default message
    - tuple[type[Exception], ...]: Only catch these exception types with default message
    - Callable[[Exception], str]: Custom function that returns error message
    - False: No retry, let exceptions propagate

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductReview(BaseModel):
    """Analysis of a product review."""
    rating: int | None = Field(description="The rating of the product", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment of the review")
    key_points: list[str] = Field(description="The key points of the review. Lowercase, 1-3 words each.")

agent = create_agent(
    model="gpt-5.5",
    tools=tools,
    response_format=ToolStrategy(ProductReview)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'"}]
})
result["structured_response"]
# ProductReview(rating=5, sentiment='positive', key_points=['fast shipping', 'expensive'])

#### Custom tool message content
The tool_message_content parameter allows you to customize the message that appears in the conversation history when structured output is generated:

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class MeetingAction(BaseModel):
    """Action items extracted from a meeting transcript."""
    task: str = Field(description="The specific task to be completed")
    assignee: str = Field(description="Person responsible for the task")
    priority: Literal["low", "medium", "high"] = Field(description="Priority level")

agent = create_agent(
    model="gpt-5.5",
    tools=[],
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="Action item captured and added to meeting notes!"
    )
)

agent.invoke({
    "messages": [{"role": "user", "content": "From our meeting: Sarah needs to update the project timeline as soon as possible"}]
})

#### Error handling
LangChain provides intelligent retry mechanisms to handle generating structure output via tool errors automatically.
- Multiple structure outputs error
- Schema validation error

##### Multiple structured output error
When a model incorrectly calls multiple structured output tools, the agent provides error feedback in a ToolMessage and prompts the model to retry

In [ ]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ContactInfo(BaseModel):
    name: str = Field(description="Person's name")
    email: str = Field(description="Email address")

class EventDetails(BaseModel):
    event_name: str = Field(description="Name of the event")
    date: str = Field(description="Event date")

agent = create_agent(
    model="gpt-5.5",
    tools=[],
    response_format=ToolStrategy(Union[ContactInfo, EventDetails])  # Default: handle_errors=True
)

agent.invoke({
    "messages": [{"role": "user", "content": "Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th"}]
})

##### Schema validation error
When structured output doesn’t match the expected schema, the agent provides specific error feedback:

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductRating(BaseModel):
    rating: int | None = Field(description="Rating from 1-5", ge=1, le=5)
    comment: str = Field(description="Review comment")

agent = create_agent(
    model="gpt-5.5",
    tools=[],
    response_format=ToolStrategy(ProductRating),  # Default: handle_errors=True
    system_prompt="You are a helpful assistant that parses product reviews. Do not make any field or value up."
)

agent.invoke({
    "messages": [{"role": "user", "content": "Parse this: Amazing product, 10/10!"}]
})

##### Error handling strategies
You can customize how errors are handled using the handle_errors parameter:
- custom error message: If handle_errors is a string, the agent will always prompt the model to re-try with a fixed tool message:
- Handle specific exception only: If handle_errors is an exception type, the agent will only retry (using the default error message) if the exception raised is the specified type. In all other cases, the exception will be raised.
- Handle multiple exception types: If handle_errors is a tuple of exceptions, the agent will only retry (using the default error message) if the exception raised is one of the specified types. In all other cases, the exception will be raised.
- custom error handler function


In [ ]:
# custom error message
ToolStrategy(
    schema=ProductRating,
    handle_errors="Please provide a valid rating between 1-5 and include a comment."
)

#Handle specific exceptions only
ToolStrategy(
    schema=ProductRating,
    handle_errors=ValueError  # Only retry on ValueError, raise others
)

#Handle multiple exception types
ToolStrategy(
    schema=ProductRating,
    handle_errors=(ValueError, TypeError)  # Retry on ValueError and TypeError
)

#Custom error handler function
from langchain.agents.structured_output import StructuredOutputValidationError
from langchain.agents.structured_output import MultipleStructuredOutputsError

def custom_error_handler(error: Exception) -> str:
    if isinstance(error, StructuredOutputValidationError):
        return "There was an issue with the format. Try again."
    elif isinstance(error, MultipleStructuredOutputsError):
        return "Multiple structured outputs were returned. Pick the most relevant one."
    else:
        return f"Error: {str(error)}"


agent = create_agent(
    model="gpt-5.5",
    tools=[],
    response_format=ToolStrategy(
                        schema=Union[ContactInfo, EventDetails],
                        handle_errors=custom_error_handler
                    )  # Default: handle_errors=True
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th"}]
})

for msg in result['messages']:
    # If message is actually a ToolMessage object (not a dict), check its class name
    if type(msg).__name__ == "ToolMessage":
        print(msg.content)
    # If message is a dictionary or you want a fallback
    elif isinstance(msg, dict) and msg.get('tool_call_id'):
        print(msg['content'])